# Pseudo-Mask Generation — Stage0 → Stage1 → Stage2


In [ ]:
# Reference: https://www.kaggle.com/code/hengck23/demo-submission
!pip install --no-deps segmentation-models-pytorch==0.5.0
!pip install connected-components-3d --no-index --find-links=file:///kaggle/input/hengck23-demo-submit-physionet/setup/

In [ ]:
import cc3d
import cv2
import pandas as pd
import numpy as np
from scipy import signal
import torch
import matplotlib.pyplot as plt
import matplotlib
import shutil
import copy
import multiprocessing as mp
import pickle
import os
import sys
from timeit import default_timer as timer
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

sys.path.insert(0, '/kaggle/input/my-stage2-lead-model')
sys.path.append('/kaggle/input/hengck23-demo-submit-physionet')
sys.path.append('/kaggle/input/physionet-final-submission-models')

from stage2_smp_model import Net as WholeModel
from stage2_lead_model import Net as LeadModel
from stage2_common import *
from stage2_model import prob_to_series_by_max
print('import ok!!!')

In [ ]:
MODE        = 'local'  # 'submit' | 'local' | 'fake'
DEVICE      = 'cuda'
FLOAT_TYPE  = torch.float16
FAIL_ID     = []

# Training config
EPOCHS        = 8
LR            = 1e-4
BATCH_SIZE    = 1
FREEZE_EPOCHS = 2
WINDOW_SIZE   = 240
OFFSET        = 416

SAVE_DIR   = '/kaggle/working/checkpoints'
KAGGLE_DIR = '/kaggle/input/physionet-ecg-image-digitization'
WEIGHT_DIR = '/kaggle/input/hengck23-demo-submit-physionet/weight'
OUT_DIR    = '/kaggle/working/output'

os.makedirs(SAVE_DIR,                exist_ok=True)
os.makedirs(f'{OUT_DIR}/rectified', exist_ok=True)
os.makedirs(f'{OUT_DIR}/masks',     exist_ok=True)

In [ ]:
def make_test_fake_df():
    valid_df = pd.read_csv(f'{KAGGLE_DIR}/train.csv')
    valid_df.loc[:,'id'] = valid_df['id'].astype(str)
    fake_test_df = []
    for i, d in valid_df.iterrows():
        image_id = d['id']
        truth_df = pd.read_csv(f'{KAGGLE_DIR}/train/{image_id}/{image_id}.csv')
        non_nan_count = truth_df.count()
        this_df = pd.DataFrame({
            'id': image_id,
            'lead': non_nan_count.index,
            'fs': d['fs'],
            'number_of_rows': non_nan_count.values
        })
        fake_test_df.append(this_df)
    return pd.concat(fake_test_df)

In [ ]:
# Set valid/test data
if MODE == 'local':
    from sample_list import ERROR_ID
    valid_df = pd.read_csv(f'{KAGGLE_DIR}/train.csv')
    valid_df['id'] = valid_df['id'].astype(str)
    valid_id = [
        f'{image_id}-{type_id}'
        for image_id in valid_df['id'].values
        for type_id in ['0001', '0003', '0004', '0005', '0006', '0009', '0010', '0011', '0012']
    ]

if MODE == 'submit':
    valid_df = pd.read_csv(f'{KAGGLE_DIR}/test.csv')
    valid_df['id'] = valid_df['id'].astype(str)
    valid_id = valid_df['id'].unique().tolist()

if MODE == 'fake':
    valid_df = make_test_fake_df()
    valid_df['id'] = valid_df['id'].astype(str)
    valid_id = valid_df['id'].unique().tolist()

print('valid_id:', len(valid_id))

In [ ]:
def read_image(sample_id):
    if MODE == 'local':
        image_id, type_id = sample_id.split('-')
        return cv2.imread(f'{KAGGLE_DIR}/train/{image_id}/{image_id}-{type_id}.png', cv2.IMREAD_COLOR_RGB)
    if MODE == 'submit':
        return cv2.imread(f'{KAGGLE_DIR}/test/{sample_id}.png', cv2.IMREAD_COLOR_RGB)
    if MODE == 'fake':
        image_id = sample_id
        type_id = ['0001', '0003', '0004', '0005', '0006', '0009', '0010', '0011', '0012'][int(image_id) % 9]
        return cv2.imread(f'{KAGGLE_DIR}/train/{image_id}/{image_id}-{type_id}.png', cv2.IMREAD_COLOR_RGB)

def read_sampling_length(sample_id):
    if MODE == 'local':
        image_id, type_id = sample_id.split('-')
        return valid_df[valid_df['id'] == image_id].iloc[0].sig_len
    d = valid_df[(valid_df['id'] == sample_id) & (valid_df['lead'] == 'II')].iloc[0]
    return d.number_of_rows

In [ ]:
def save_sparse_mask_coo(mask_dense, save_path):
    """mask_dense: (4, H, W) float32 — saves non-zero entries in COO format."""
    shape = np.array(mask_dense.shape)
    arrays = {'shape': shape}
    for i in range(mask_dense.shape[0]):
        ys, xs = np.where(mask_dense[i] > 0.3)
        vs = mask_dense[i, ys, xs]
        arrays[f'ch{i}_y'] = ys.astype(np.int32)
        arrays[f'ch{i}_x'] = xs.astype(np.int32)
        arrays[f'ch{i}_v'] = vs.astype(np.float32)
    np.savez_compressed(save_path, **arrays)

def load_sparse_mask_coo(filepath):
    """Returns (4, H, W) float32."""
    data = np.load(filepath)
    shape = tuple(data['shape'])
    mask = np.zeros(shape, dtype=np.float32)
    for i in range(shape[0]):
        y, x, v = data[f'ch{i}_y'], data[f'ch{i}_x'], data[f'ch{i}_v']
        mask[i, y, x] = v
    return mask

In [ ]:
xscale = 5000 / (2080 - 118)
addx   = 1
yscale = 1
IMGH, IMGW = int(1700 * yscale), int(2200 * xscale + addx)
x0, x1 = 0, 5600
y0, y1 = 0, 1696
zero_mv = [703.5, 987.5, 1271.5, 1531.5]

def read_images(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMGW, IMGH), interpolation=cv2.INTER_LINEAR)
    trim_image = image.copy()[OFFSET:y1, x0:x1]
    image = image[y0:y1, x0:x1]
    H, W, _ = image.shape
    lead_images = []
    for zmv in zero_mv:
        h0, h1 = int(zmv - WINDOW_SIZE), int(zmv + WINDOW_SIZE)
        src_h0, src_h1 = max(0, h0), min(H, h1)
        dst_h0 = src_h0 - h0
        dst_h1 = dst_h0 + (src_h1 - src_h0)
        lead_img = np.zeros((WINDOW_SIZE * 2, W, 3), np.uint8)
        lead_img[dst_h0:dst_h1] = image[src_h0:src_h1]
        lead_images.append(lead_img)
    return trim_image, np.stack(lead_images)

In [ ]:
from stage0_common import time_to_str
from stage0_model  import Net as Stage0Net
from stage1_model  import Net as Stage1Net
import stage0_common as s0c
import stage1_common as s1c

# Load all three models once
stage0_net = Stage0Net(pretrained=False)
stage0_net = load_net(stage0_net, f'{WEIGHT_DIR}/stage0-last.checkpoint.pth')
stage0_net.to(DEVICE).eval()

stage1_net = Stage1Net(pretrained=False)
stage1_net = load_net(stage1_net, f'{WEIGHT_DIR}/stage1-last.checkpoint.pth')
stage1_net.to(DEVICE).eval()

mask_model = LeadModel(
    encoder_name='tu-timm/tf_efficientnet_b6.ns_jft_in1k',
    encoder_weights=None,
    fusion_type='shared_conv2d',
)
state = torch.load(
    '/kaggle/input/physionet-final-submission-models/series_b6_shared_conv2d_lb23.10.pth',
    map_location='cpu'
)
mask_model.load_state_dict(state, strict=False)
mask_model.to(DEVICE).eval()
mask_model.output_type = ['infer']

In [ ]:
start_timer = timer()

for n, sample_id in enumerate(valid_id):
    timestamp = time_to_str(timer() - start_timer, 'sec')
    print(f'\r\t {n:4d} {sample_id}', timestamp, end='', flush=True)

    # ── Stage 0: rectification ──
    try:
        image = read_image(sample_id)
        batch = s0c.image_to_batch(image)
        with torch.amp.autocast('cuda', dtype=FLOAT_TYPE):
            with torch.no_grad():
                output0 = stage0_net(batch)
        rotated, keypoint = s0c.output_to_predict(image, batch, output0)
        normalised, keypoint, homo = s0c.normalise_by_homography(rotated, keypoint)
    except Exception as e:
        print(f'\nStage0 failed {sample_id}: {e}')
        FAIL_ID.append(sample_id)
        torch.cuda.empty_cache()
        continue

    # ── Stage 1: grid-point detection ──
    try:
        batch1 = {'image': torch.from_numpy(np.ascontiguousarray(normalised.transpose(2, 0, 1))).unsqueeze(0)}
        with torch.amp.autocast('cuda', dtype=FLOAT_TYPE):
            with torch.no_grad():
                output1 = stage1_net(batch1)
        gridpoint_xy, more = s1c.output_to_predict(normalised, batch1, output1)
        rectified = s1c.rectify_image(normalised, gridpoint_xy)
        cv2.imwrite(
            f'{OUT_DIR}/rectified/{sample_id}.rect.jpg',
            cv2.cvtColor(rectified, cv2.COLOR_RGB2BGR),
            [int(cv2.IMWRITE_JPEG_QUALITY), 70]
        )
    except Exception as e:
        print(f'\nStage1 failed {sample_id}: {e}')
        FAIL_ID.append(sample_id)
        torch.cuda.empty_cache()
        continue

    # ── Stage 2: pseudo-mask generation ──
    try:
        _, lead_images = read_images(f'{OUT_DIR}/rectified/{sample_id}.rect.jpg')
        lead_tensor = torch.from_numpy(
            lead_images.transpose(0, 3, 1, 2)
        ).contiguous().unsqueeze(0).to(DEVICE)
        with torch.amp.autocast('cuda', dtype=FLOAT_TYPE):
            with torch.no_grad():
                output2 = mask_model({'image': lead_tensor})
        mask_dense = output2['pixel'].squeeze(0).squeeze(1).float().cpu().numpy()
        save_sparse_mask_coo(mask_dense, f'{OUT_DIR}/masks/{sample_id}.mask-coo.npz')
    except Exception as e:
        print(f'\nMask failed {sample_id}: {e}')

    torch.cuda.empty_cache()

print('')
print('Pipeline done. FAIL_ID:', FAIL_ID)